# ColorCorrect: compare fit methods and neural options

Compare five CC families (six method choices), edit neural-network settings, and
read fitted-chart error alongside CC-only validation. Install the SDK wheel and
Jupyter, then set `COLORCORRECT_APP_ROOT`, `COLORCORRECT_SAMPLES`, and
`COLORCORRECT_OUTPUT_DIR` before starting the kernel. Each run saves its models
and results in a new subdirectory. With the 0.2.2 engine, Auto chart
and white-plane detection try an eligible GPU and recover to CPU when needed;
fitting uses the CPU. Older 0.2.1 packages still require GPU inference.

See [notebook setup](README.md), [configuration reference](../docs/api/README.md#configuration-reference),
[ColorCorrectionPipeline](https://pypi.org/project/ColorCorrectionPipeline/), and
[the original study](https://doi.org/10.1002/ppj2.70067).

## 1. Setup

The same installed SDK, chart, blank-field photograph, and one immutable detection are used for every fit.

In [ ]:
import json, os, sys, tempfile
from pathlib import Path
import colorcorrect
from colorcorrect import (
    ApplyOptions, CalibrationOptions, Client, ColorCorrection, Config, FFCOptions,
    NeuralNetworkOptions, ReferenceProfile,
)

def required_directory(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} to an explicit directory before starting the kernel")
    path = Path(value).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"{variable}: {path}")
    return path

app = required_directory("COLORCORRECT_APP_ROOT")
samples = required_directory("COLORCORRECT_SAMPLES")
output_root = required_directory("COLORCORRECT_OUTPUT_DIR")
module_path = Path(colorcorrect.__file__).resolve()
assert module_path.is_relative_to(Path(sys.prefix).resolve()), "Install the wheel in this kernel environment; do not import from a checkout"
chart, white = samples / "ColorCard_Big.JPG", samples / "Blank.JPG"
scenes = [samples / "Sample_3572.JPG", samples / "Sample_3573.JPG"]
assert all(path.is_file() for path in [chart, white, *scenes]), "The public sample set is incomplete"
out = Path(tempfile.mkdtemp(prefix="tutorial-", dir=output_root))
print("SDK:", colorcorrect.__version__, module_path)
print("Native app:", app, "\nArtifacts:", out)

from dataclasses import replace
from time import perf_counter
rows = []

## 2. Compare five families

`ccm` is a matrix; `linear` selects polynomial regression; `rpcc2`/`rpcc3` are two orders of root-polynomial; `pls` is partial least squares; `nn` is a neural fit. The polynomial degree does not control GC or RPCC. Keep FFC/GC/WB fixed. Each timing includes loading, fitting, CC-only validation, and serialization, but excludes detection and final output encoding. Caches and worker initialization can affect timing; these observations are not a controlled benchmark.

In [ ]:
base = CalibrationOptions(
    do_ffc=True, do_gc=True, do_wb=True, do_cc=True,
    ffc=FFCOptions(method="pls", bins=32, smooth=5, degree=3),
    write_output=False, cross_validate=True,
    nn=NeuralNetworkOptions(hidden_layers=(64, 32), epochs=200, patience=50),
)
with Client(root=app, backend="cpu", timeout=900) as client:
    worker = client.engine.executable.resolve()
    assert worker.is_relative_to(app)
    detection = client.detect(chart)
    for method in ("ccm", "linear", "rpcc2", "rpcc3", "pls", "nn"):
        options = replace(base, model=method, degree=2, pls_components=0)
        started = perf_counter()
        fitted = client.calibrate(chart, white=white, detection=detection,
            output_dir=out / "methods", options=options)
        report = fitted.metrics
        assert report.get("do_ffc") is True, fitted.warnings
        row = {"method": method, "seconds": round(perf_counter() - started, 3),
               "training": report["delta_e_2000"]["mean_after"],
               "cc_only_validation": report.get("validation_delta_e_2000"),
               "model": str(fitted.path), "warnings": list(fitted.warnings),
               "backends": {"actual": fitted.backends.actual, "diagnostics": fitted.backends.diagnostics}}
        rows.append(row)
        print(json.dumps(row))
assert len(rows) == 6

## 3. Edit neural fitting options

These are CC neural options, separate from FFC `nn_max_iter`. Compare Adam, SGD, and RMSprop with identical topology/budget: two layers, batch normalization, dropout, learning rate, and early stopping. The 200-epoch tutorial budget demonstrates the interface; it is not a production recommendation. Detection uses a separate setting: `backend="cpu-inference"` explicitly selects CPU detection with a capable worker; legacy `backend="cpu"` retains Auto detection.

In [ ]:
with Client(root=app, backend="cpu", timeout=900) as client:
    for optimizer in ("adam", "sgd", "rmsprop"):
        options = replace(base, model="nn", nn=NeuralNetworkOptions(
            hidden_layers=(64, 32), optimizer=optimizer, batch_norm=True,
            dropout=0.1, epochs=200, patience=50, lr=0.001, tol=1e-6,
        ))
        fitted = client.calibrate(chart, white=white, detection=detection,
            output_dir=out / "optimizers", options=options)
        report = fitted.metrics
        print(optimizer, "training:", report["delta_e_2000"]["mean_after"],
              "CC-only validation:", report.get("validation_delta_e_2000"),
              "warnings:", fitted.warnings)
    final_model = fitted.path
    # Explicit rendering settings are used by this advanced raw preview call.
    client.engine.preview(final_model, chart, "cc", out / "nn-preview", do_ffc=True,
        white=white, ffc=base.ffc.to_dict(), fast_mode=True)

## 4. Compare the images

Display the final RMSprop example beside the original. This illustrates one
configuration, rather than identifying the best optimizer. The preview uses
the same white-image geometry and FFC settings as the fit. PNG thumbnails avoid
additional JPEG quantization.

In [ ]:
from io import BytesIO
from PIL import Image as PillowImage
from IPython.display import Image, Markdown, display
for path, caption in [(chart, "Original"), (out / "nn-preview" / "corrected.png", "RMSprop example")]:
    assert path.is_file(), path
    with PillowImage.open(path) as original:
        thumb = original.convert("RGB")
        thumb.thumbnail((640, 420))
        buffer = BytesIO()
        thumb.save(buffer, format="PNG")
    display(Markdown("**" + caption + "**"), Image(data=buffer.getvalue()))

## Try another configuration

The final cell orders methods by fitted-chart error. Compare their CC-only
validation scores and warnings as well, then assess a separately captured
photograph. Try `replace(base, model="linear", degree=3)` or
`replace(base, do_wb=False)` in a new fit, changing one setting at a time.
Detect again if you replace the chart photograph.

In [ ]:
print("Training-only ranking (not a recommendation):")
for row in sorted(rows, key=lambda r: r["training"]):
    print(row["method"], round(row["training"], 3), "seconds:", row["seconds"])
(out / "tutorial-result.json").write_text(json.dumps({"ok": True, "methods": rows,
    "worker": str(worker), "kernel_interpreter": sys.executable, "sdk_module": str(module_path), "final_model": str(final_model)}, indent=2), encoding="utf-8")
print("Durable artifacts:", out)